In [1]:
import pandas as pd
import numpy as np
#read in the data from the csv file
df = pd.read_csv('SGJobDataCopy.csv' ,nrows=1200000)


In [2]:
#copy the main data into draft and keep the orignal data intact in df
draft=df.copy()

draft.head()
#draft.info()
#convert the expiryDate,originalPostingDate,newPostingDate column to datetime format
draft["metadata_expiryDate"] = pd.to_datetime(draft["metadata_expiryDate"], dayfirst=True)
draft["metadata_originalPostingDate"] = pd.to_datetime(draft["metadata_originalPostingDate"], dayfirst=True)
draft["metadata_newPostingDate"] = pd.to_datetime(draft["metadata_newPostingDate"], dayfirst=True)
draft.info()

print("Minimum original posting date:", draft["metadata_originalPostingDate"].min())
print("Maximum original posting date:", draft["metadata_originalPostingDate"].max())
#Date range of the data is from 03-10-2022 to 29-05-2024

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   categories                          1044597 non-null  object        
 1   employmentTypes                     1044597 non-null  object        
 2   metadata_expiryDate                 1044597 non-null  datetime64[ns]
 3   metadata_isPostedOnBehalf           1048585 non-null  bool          
 4   metadata_jobPostId                  1044597 non-null  object        
 5   metadata_newPostingDate             1044597 non-null  datetime64[ns]
 6   metadata_originalPostingDate        1044597 non-null  datetime64[ns]
 7   metadata_repostCount                1048585 non-null  int64         
 8   metadata_totalNumberJobApplication  1048585 non-null  int64         
 9   metadata_totalNumberOfView          1048585 non-null  int64         

In [ ]:
draft = draft.drop_duplicates()
draft.duplicated().sum()
#draft["employmentTypes"].value_counts()
#draft["employmentTypes"].isna().sum()
#Unique values in employment types and no NaN values in employment types


0

In [4]:
draft["positionLevels"].value_counts()
draft["positionLevels"].isna().sum()
draft.loc[draft["positionLevels"].isna()]
draft = draft.drop(draft[draft["positionLevels"].isna()].index)
draft["positionLevels"].isna().sum()
#Dropped positionLevels with NaN values


0

In [5]:
draft["salary_type"].value_counts()
draft["salary_type"].isna().sum()
#No NaN values in salary_type

0

In [6]:
draft.describe()
# For rows that have value of less than 100 for maximum,minimum,average salary for monthly salary, replaced the value with NaN as 
# it is not possible to have a salary of 100 for monthly salary. This is done to clean the data and avoid any bias in the analysis.
draft.loc[
    (draft["salary_maximum"] < 100) |
    (draft["salary_minimum"] < 100) |
    (draft["average_salary"] < 100),
    ["salary_maximum", "salary_minimum", "average_salary"]
] = np.nan

# In Cases where the average monthly salary is more than 375,500 for Non executive positions, replaced the value with NaN as it is not possible to have a salary of 375,500 for Non executive positions. This is done to clean the data and avoid any bias in the analysis.
draft.loc[
    (draft["average_salary"] > 375500.0),
    ["salary_maximum", "salary_minimum", "average_salary"]
] = np.nan

#draft["salary_maximum"].describe()
draft["salary_minimum"].describe()
#draft.nlargest(20, "average_salary")


count    1.037163e+06
mean     3.855218e+03
std      3.085934e+03
min      1.000000e+02
25%      2.500000e+03
50%      3.000000e+03
75%      4.500000e+03
max      3.500000e+05
Name: salary_minimum, dtype: float64

In [7]:
#Split the Categories Column into 2 Columns which contains ID and Category
import re

draft["id"] = draft["categories"].str.findall( r'"id"\s*:\s*(\d+)').apply(", ".join)
draft["category"] = draft["categories"].str.findall(r'"category"\s*:\s*"([^"]+)"').apply(", ".join)



In [ ]:
#Package a 1 year subset of the cleaned data for visualization and analysis. The subset will contain data from 2023 only.
clean_data_2023 = draft.loc[draft["metadata_originalPostingDate"].between("2023-01-01", "2023-12-31")]
clean_data_2023.to_csv("clean_data_2023.csv", index=False)

#Package the complete set for visualization and analysis in Streamlit.
#draft.to_csv("draft.csv", index=False)

In [8]:
#Find the Job Category with the most number of Job Postings
job_counts = (
    draft.groupby("category")
      .size()
      .reset_index(name="vacancies")
      .sort_values("vacancies", ascending=False)
)
print("Job Category with the most number of Job Postings from 03-10-2022 to 29-05-2024:")
print(job_counts)
print("Job Category with the most number of Job Postings is:", job_counts.iloc[0]["category"], "with", job_counts.iloc[0]["vacancies"], "vacancies."  )

Job Category with the most number of Job Postings from 03-10-2022 to 29-05-2024:
                                                category  vacancies
20289                             Information Technology      92869
16758                                        Engineering      49494
18699                                                F&B      48461
0                       Accounting / Auditing / Taxation      44720
9018                           Building and Construction      44374
...                                                  ...        ...
6439   Advertising / Media, Customer Service, Enterta...          1
13773  Customer Service, Entertainment, Hospitality, ...          1
13774   Customer Service, Entertainment, Human Resources          1
13775  Customer Service, Entertainment, Information T...          1
10562  Building and Construction, General Work, Repai...          1

[21125 rows x 2 columns]
Job Category with the most number of Job Postings is: Information Technology 

In [9]:
#To find the median salary for a particular Job Category
Category_Summary = (
    draft[draft["category"] == "Design"] # Edit to include requested Category
      .groupby("positionLevels")
      .agg(
          Vacancies=("average_salary", "count"),
          Median_salary=("average_salary", "median")
      )
      .reset_index()
      .sort_values("Median_salary", ascending=False)
)

print(Category_Summary)

      positionLevels  Vacancies  Median_salary
8  Senior Management         49        10500.0
3            Manager        159         6500.0
4  Middle Management         74         6325.0
6       Professional        288         5250.0
7   Senior Executive        554         4500.0
0          Executive        916         3500.0
2   Junior Executive        767         3250.0
5      Non-executive        232         3000.0
1  Fresh/entry level        270         2250.0


In [12]:
summary = (
    draft.groupby(["category", "positionLevels"])
      .agg(
          vacancies=("average_salary", "count"),
          median_salary=("average_salary", "median")
      )
      .reset_index()
      .sort_values("median_salary", ascending=False)
)

summary.head()

,category,positionLevels,vacancies,median_salary
29570,"Customer Service, Information Technology, Sale...",Senior Management,1,375000.0
42508,"Sales / Retail, Security and Investigation",Senior Management,1,275000.0
42506,"Sales / Retail, Security and Investigation",Middle Management,1,250000.0
13477,"Advertising / Media, Information Technology, T...",Senior Management,1,250000.0
5916,"Admin / Secretarial, Consulting, General Manag...",Senior Management,1,210000.0


In [3]:
df.info()
df["metadata_originalPostingDate"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83668 entries, 0 to 83667
Data columns (total 23 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   employmentTypes                     83668 non-null  object 
 1   metadata_expiryDate                 83668 non-null  object 
 2   metadata_isPostedOnBehalf           83668 non-null  bool   
 3   metadata_jobPostId                  83668 non-null  object 
 4   metadata_newPostingDate             83668 non-null  object 
 5   metadata_originalPostingDate        83668 non-null  object 
 6   metadata_repostCount                83668 non-null  int64  
 7   metadata_totalNumberJobApplication  83668 non-null  int64  
 8   metadata_totalNumberOfView          83668 non-null  int64  
 9   minimumYearsExperience              83668 non-null  int64  
 10  numberOfVacancies                   83668 non-null  int64  
 11  positionLevels                      83668

count          83668
unique           206
top       2023-03-30
freq            5983
Name: metadata_originalPostingDate, dtype: object